#Brain tumor cost prediction


##Splitting

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

data = df.copy()
data.columns = data.columns.str.strip()

# 1. เลือกเฉพาะกลุ่ม Tumor
# รองรับ DISE_gr = 1 หรือ "Tumor"
disease = data['DISE_gr'].astype('string').str.strip().str.lower()

is_tumor = (
    disease.eq('tumor')
    | pd.to_numeric(data['DISE_gr'], errors='coerce').eq(1)
).fillna(False)

data = data.loc[is_tumor].copy()

# 2. ตรวจสอบ outcome และรหัสผู้ป่วย
data['TOTAL_COST'] = pd.to_numeric(
    data['TOTAL_COST'], errors='coerce'
).replace([np.inf, -np.inf], np.nan)

data['HN'] = data['HN'].astype('string').str.strip()

valid = (
    data['TOTAL_COST'].notna()
    & data['TOTAL_COST'].gt(0)
    & data['HN'].notna()
    & data['HN'].ne('')
)

print(f'ตัดรายการที่ไม่มี cost/HN หรือ cost ≤ 0: {(~valid).sum()} รายการ')

data = data.loc[valid].reset_index(drop=True)

if data['HN'].nunique() < 2:
    raise ValueError('ต้องมีผู้ป่วยอย่างน้อย 2 คนเพื่อแบ่งข้อมูล')

# 3. Random split: 70% training / 30% test ตามผู้ป่วย
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(data, groups=data['HN'])
)

train_df = data.iloc[train_idx].copy()
test_df = data.iloc[test_idx].copy()

# 4. ตรวจสอบว่าไม่มีผู้ป่วยซ้ำระหว่างสองชุด
overlap = set(train_df['HN']) & set(test_df['HN'])
assert len(overlap) == 0, 'พบผู้ป่วยซ้ำระหว่าง train และ test!'

# 5. สรุปผลการแบ่งข้อมูล
summary = pd.DataFrame({
    'Dataset': ['Total', 'Train', 'Test'],
    'Records': [len(data), len(train_df), len(test_df)],
    'Patients': [
        data['HN'].nunique(),
        train_df['HN'].nunique(),
        test_df['HN'].nunique()
    ],
    'Records (%)': [
        100,
        100 * len(train_df) / len(data),
        100 * len(test_df) / len(data)
    ]
})

summary['Records (%)'] = summary['Records (%)'].round(1)
display(summary)

print(f'ผู้ป่วยซ้ำระหว่าง train/test: {len(overlap)} คน')

# Outcome สำหรับขั้นตอนสร้างโมเดล
y_train = train_df['TOTAL_COST'].copy()
y_test = test_df['TOTAL_COST'].copy()

##Training data preparation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# =========================================================
# 1. คัดลอกเฉพาะ training set
# =========================================================
corr_train = train_df.copy().reset_index(drop=True)
corr_train.columns = corr_train.columns.str.strip()

# รองรับชื่อเดิม OP_WFME
if 'OP_TYPE' not in corr_train.columns:
    if 'OP_WFME' in corr_train.columns:
        corr_train = corr_train.rename(
            columns={'OP_WFME': 'OP_TYPE'}
        )
    else:
        raise ValueError('ไม่พบ OP_TYPE หรือ OP_WFME')


# =========================================================
# 2. กำหนดตัวแปร
# =========================================================
continuous_vars = ['age', 'Pre_KPS']

binary_vars = [
    'HT', 'DM', 'DYSLIPID',
    'LFT', 'RF', 'LUNG', 'CVA', 'IHD',
    'PRE_SEIZURE', 'PRE_WEAK',
    'Antiplatelet_Anticoagulant'
]

required_columns = (
    continuous_vars
    + binary_vars
    + [
        'gender', 'WHO_grade',
        'OP_TYPE', 'OP_time_min', 'TOTAL_COST'
    ]
)

missing_columns = [
    col for col in required_columns
    if col not in corr_train.columns
]

if missing_columns:
    raise ValueError(f'ไม่พบคอลัมน์: {missing_columns}')


# =========================================================
# 3. ฟังก์ชันแปลงข้อมูล
# =========================================================
def numeric_checked(series):
    """แปลงเป็น numeric และแจ้งเมื่อพบข้อความที่แปลงไม่ได้"""
    raw = series.replace(r'^\s*$', np.nan, regex=True)
    result = pd.to_numeric(raw, errors='coerce')

    invalid = raw.notna() & result.isna()

    if invalid.any():
        raise ValueError(
            f'{series.name}: พบค่าที่แปลงเป็นตัวเลขไม่ได้: '
            f'{raw[invalid].unique().tolist()}'
        )

    return result.replace([np.inf, -np.inf], np.nan)


def map_checked(series, mapping):
    """แปลง label หรือรหัส โดยคง missing ไว้"""
    text = (
        series.astype('string')
        .str.strip()
        .str.lower()
        .replace('', pd.NA)
    )

    result = text.map(mapping)

    invalid = text.notna() & result.isna()

    if invalid.any():
        raise ValueError(
            f'{series.name}: พบค่าที่ไม่อยู่ใน mapping: '
            f'{text[invalid].unique().tolist()}'
        )

    return result.astype(float)


# =========================================================
# 4. แปลงตัวแปรต่อเนื่องและ WHO grade
# =========================================================
numeric_vars = [
    'age', 'Pre_KPS', 'WHO_grade',
    'OP_time_min', 'TOTAL_COST'
]

for col in numeric_vars:
    corr_train[col] = numeric_checked(corr_train[col])


# =========================================================
# 5. แปลง binary variables เป็น 0/1
# =========================================================
binary_mapping = {
    'no': 0, 'yes': 1,
    '0': 0, '1': 1,
    '0.0': 0, '1.0': 1
}

for col in binary_vars:
    corr_train[col] = map_checked(
        corr_train[col],
        binary_mapping
    )


# =========================================================
# 6. Sex: 0 = male, 1 = female
# ข้อมูลต้นฉบับ gender: 1 = male, 2 = female
# =========================================================
sex_mapping = {
    'male': 0, 'female': 1,
    '1': 0, '2': 1,
    '1.0': 0, '2.0': 1
}

corr_train['sex_female'] = map_checked(
    corr_train['gender'],
    sex_mapping
)


# =========================================================
# 7. OP_TYPE เป็น numeric ตามรหัสต้นฉบับ
# =========================================================
op_raw = (
    corr_train['OP_TYPE']
    .astype('string')
    .str.strip()
    .replace('', pd.NA)
)

op_numeric = pd.to_numeric(op_raw, errors='coerce')

# หาก OP_TYPE เป็นชื่อชนิดผ่าตัด ให้ใส่ mapping
# ตามรหัสต้นฉบับของคุณ ห้ามใช้รหัสตัวอย่างแทนโดยไม่ตรวจสอบ
OP_TYPE_MAPPING = {
    # 'ชื่อชนิดผ่าตัด': รหัสตัวเลข,
}

needs_mapping = op_raw.notna() & op_numeric.isna()

if needs_mapping.any():

    if not OP_TYPE_MAPPING:
        raise ValueError(
            'OP_TYPE เป็นข้อความ กรุณากำหนด OP_TYPE_MAPPING '
            'ตามรหัสต้นฉบับ แล้วรันใหม่\n'
            f'ชื่อที่ต้องกำหนดรหัส: '
            f'{op_raw[needs_mapping].unique().tolist()}'
        )

    # ทำให้การจับคู่ชื่อไม่ขึ้นกับตัวพิมพ์ใหญ่/เล็ก
    mapping_normalized = {
        str(key).strip().lower(): value
        for key, value in OP_TYPE_MAPPING.items()
    }

    mapped = pd.to_numeric(
        op_raw.str.lower().map(mapping_normalized),
        errors='coerce'
    )

    op_numeric = op_numeric.fillna(mapped)

    unknown = op_raw.notna() & op_numeric.isna()

    if unknown.any():
        raise ValueError(
            'ยังไม่ได้กำหนดรหัสสำหรับ: '
            f'{op_raw[unknown].unique().tolist()}'
        )

corr_train['OP_TYPE'] = (
    op_numeric.astype(float)
    .replace([np.inf, -np.inf], np.nan)
)


# =========================================================
# 8. จัดลำดับ matrix — TOTAL_COST อยู่สุดท้าย
# =========================================================
correlation_vars = [
    'age',
    'sex_female',
    'Pre_KPS',
    'HT',
    'DM',
    'DYSLIPID',
    'LFT',
    'RF',
    'LUNG',
    'CVA',
    'IHD',
    'PRE_SEIZURE',
    'PRE_WEAK',
    'Antiplatelet_Anticoagulant',
    'WHO_grade',
    'OP_TYPE',
    'OP_time_min',
    'TOTAL_COST'
]

corr_data = corr_train[correlation_vars].astype(float)

assert corr_data.columns[-1] == 'TOTAL_COST'


# =========================================================
# 9. ตรวจสอบข้อมูลก่อน correlation
# =========================================================
data_check = pd.DataFrame({
    'Non_missing': corr_data.notna().sum(),
    'Missing': corr_data.isna().sum(),
    'Unique_values': corr_data.nunique(dropna=True),
    'Minimum': corr_data.min(),
    'Maximum': corr_data.max()
})

print(f'Training records: {len(corr_data)}')
display(data_check)

constant_columns = data_check.index[
    data_check['Unique_values'] < 2
].tolist()

if constant_columns:
    print(
        'ตัวแปรเหล่านี้มีข้อมูลไม่พอหรือไม่มีความแปรปรวน '
        'ค่า correlation จะแสดงเป็นช่องว่าง:',
        constant_columns
    )

print('พร้อมทำ correlation โดยใช้ corr_data')

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

# =====================================================
# SETTINGS
# =====================================================

# True = เพิ่มระยะเวลาผ่าตัดจริง
# โมเดลจะไม่ใช่ preoperative prediction
INCLUDE_OP_TIME = False

# True = เพิ่ม WHO grade เป็น categorical
# ตรวจสอบความถูกต้องและเวลาที่ทราบ grade ก่อนใช้
INCLUDE_WHO_GRADE = False

SCREEN_P = 0.20

# บังคับชนิดผ่าตัด ไม่ให้ backward ตัดออก
FORCED_FEATURES = {'OP_TYPE'}

# หากต้องการบังคับอายุด้วย ให้ใช้:
# FORCED_FEATURES = {'OP_TYPE', 'age'}

# ใช้เฉพาะ training
reg_train = corr_train.copy().reset_index(drop=True)

numeric_features = ['age', 'Pre_KPS']

binary_features = [
    'sex_female',
    'HT', 'DM', 'DYSLIPID',
    'LFT', 'RF', 'LUNG', 'CVA', 'IHD',
    'PRE_SEIZURE', 'PRE_WEAK',
    'Antiplatelet_Anticoagulant'
]

categorical_features = ['OP_TYPE']

if INCLUDE_OP_TIME:
    numeric_features.append('OP_time_min')

if INCLUDE_WHO_GRADE:
    categorical_features.append('WHO_grade')

candidate_features = (
    numeric_features
    + binary_features
    + categorical_features
)

# ตรวจสอบคอลัมน์
required = candidate_features + ['TOTAL_COST', 'HN']
missing_columns = [c for c in required if c not in reg_train]

if missing_columns:
    raise ValueError(f'ไม่พบคอลัมน์: {missing_columns}')

# ข้อมูลใน corr_train ผ่านการแปลงเป็น numeric แล้ว
for col in candidate_features + ['TOTAL_COST']:
    raw = reg_train[col].replace(r'^\s*$', np.nan, regex=True)
    converted = pd.to_numeric(raw, errors='coerce')

    if (raw.notna() & converted.isna()).any():
        raise ValueError(f'{col}: กรุณาเตรียมข้อมูล numeric ก่อน')

    reg_train[col] = converted.replace(
        [np.inf, -np.inf], np.nan
    )

reg_train['HN'] = reg_train['HN'].astype('string').str.strip()

if (
    reg_train['TOTAL_COST'].isna().any()
    or reg_train['HN'].isna().any()
    or reg_train['HN'].eq('').any()
):
    raise ValueError('พบ missing TOTAL_COST หรือ HN')

for col in binary_features:
    if not set(reg_train[col].dropna().unique()).issubset({0, 1}):
        raise ValueError(f'{col} ต้องเป็น 0/1')

# เติม missing จาก training เท่านั้น
imputation_values = {}

for col in candidate_features:

    observed = reg_train[col].dropna()

    if observed.empty:
        raise ValueError(f'{col}: ไม่มีข้อมูลใน training')

    if col in numeric_features:
        fill_value = float(observed.median())
    else:
        fill_value = float(observed.mode().iloc[0])

    imputation_values[col] = fill_value
    reg_train[col] = reg_train[col].fillna(fill_value)

# ตัดตัวแปรที่ไม่มีความแปรปรวน
constant_features = [
    col for col in candidate_features
    if reg_train[col].nunique() < 2
]

if FORCED_FEATURES.intersection(constant_features):
    raise ValueError(
        'ตัวแปรบังคับไม่มีความแปรปรวน: '
        f'{FORCED_FEATURES.intersection(constant_features)}'
    )

candidate_features = [
    col for col in candidate_features
    if col not in constant_features
]

if not FORCED_FEATURES.issubset(candidate_features):
    raise ValueError('ตัวแปรบังคับต้องอยู่ใน candidate_features')

print('Excluded constant features:', constant_features)
print('Candidate features:', candidate_features)

In [ ]:
X_all = pd.DataFrame(index=reg_train.index)

feature_blocks = {}
category_levels = {}

for feature in candidate_features:

    if feature in categorical_features:

        levels = sorted(
            reg_train[feature].unique().tolist()
        )

        category_levels[feature] = levels

        category = pd.Series(
            pd.Categorical(
                reg_train[feature],
                categories=levels
            ),
            index=reg_train.index
        )

        dummies = pd.get_dummies(
            category,
            prefix=feature,
            drop_first=True,
            dtype=float
        )

        X_all = pd.concat([X_all, dummies], axis=1)
        feature_blocks[feature] = dummies.columns.tolist()

        print(f'{feature}: reference = {levels[0]}')

    else:

        X_all[feature] = reg_train[feature].astype(float)
        feature_blocks[feature] = [feature]

y = reg_train['TOTAL_COST'].astype(float)
groups = reg_train['HN'].astype(str)

if groups.nunique() < 2:
    raise ValueError('ต้องมีผู้ป่วยอย่างน้อย 2 คน')


def make_design(features):
    columns = [
        column
        for feature in features
        for column in feature_blocks[feature]
    ]

    X = X_all[columns].copy()
    X.insert(0, 'const', 1.0)

    return X.astype(float)


def fit_model(features):
    X = make_design(features)

    if len(X) <= X.shape[1]:
        raise ValueError('จำนวนตัวอย่างไม่เพียงพอ')

    if np.linalg.matrix_rank(X.to_numpy()) < X.shape[1]:
        raise ValueError(
            f'Design matrix มีตัวแปรซ้ำซ้อน: {features}'
        )

    # Cluster-robust SE สำหรับข้อมูลซ้ำในผู้ป่วยคนเดียวกัน
    return sm.OLS(y, X).fit(
        cov_type='cluster',
        cov_kwds={'groups': groups},
        use_t=True
    )


def coefficient_table(model):
    ci = model.conf_int()

    return pd.DataFrame({
        'Term': model.params.index,
        'Beta': model.params.values,
        'Cluster_robust_SE': model.bse.values,
        'CI95_low': ci.iloc[:, 0].values,
        'CI95_high': ci.iloc[:, 1].values,
        'p_value': model.pvalues.values
    })

#Simple linear regression และ screening

In [ ]:
screening_rows = []
simple_tables = []

for feature in candidate_features:

    model = fit_model([feature])

    # ทดสอบทุก coefficient ยกเว้น intercept
    restriction = np.eye(len(model.params))[1:, :]

    overall_p = float(
        model.wald_test(
            restriction,
            use_f=True,
            scalar=True
        ).pvalue
    )

    screening_rows.append({
        'Predictor': feature,
        'Overall_p': overall_p,
        'Training_R2': model.rsquared,
        'Pass_screen': overall_p < SCREEN_P,
        'Forced_in': feature in FORCED_FEATURES
    })

    table = coefficient_table(model)
    table.insert(0, 'Predictor', feature)
    table['Overall_predictor_p'] = overall_p

    simple_tables.append(table)

screening_table = pd.DataFrame(screening_rows)

simple_results = pd.concat(
    simple_tables,
    ignore_index=True
)

display(screening_table.round(4))
display(simple_results.round(4))

screened_features = screening_table.loc[
    screening_table['Pass_screen']
    | screening_table['Forced_in'],
    'Predictor'
].tolist()

print('Features entering multiple regression:')
print(screened_features)

screening_table.to_csv(
    'screening_results.csv',
    index=False,
    encoding='utf-8-sig'
)

simple_results.to_csv(
    'simple_regression_results.csv',
    index=False,
    encoding='utf-8-sig'
)

#Backward AIC

In [ ]:
selected_features = screened_features.copy()
selected_model = fit_model(selected_features)

selection_history = [{
    'Step': 0,
    'Removed': 'START',
    'AIC': selected_model.aic,
    'Features': ', '.join(selected_features)
}]

all_trials = []
step = 0

while True:

    removable_features = [
        feature for feature in selected_features
        if feature not in FORCED_FEATURES
    ]

    if not removable_features:
        break

    trials = []

    for feature in removable_features:

        trial_features = [
            f for f in selected_features
            if f != feature
        ]

        trial_model = fit_model(trial_features)

        trials.append({
            'removed': feature,
            'features': trial_features,
            'model': trial_model,
            'aic': trial_model.aic
        })

        all_trials.append({
            'Step': step + 1,
            'Candidate_removed': feature,
            'Current_AIC': selected_model.aic,
            'Candidate_AIC': trial_model.aic
        })

    best_trial = min(
        trials,
        key=lambda item: item['aic']
    )

    # หยุดเมื่อการตัดเพิ่มเติมไม่ลด AIC
    if best_trial['aic'] >= selected_model.aic - 1e-8:
        break

    selected_features = best_trial['features']
    selected_model = best_trial['model']
    step += 1

    selection_history.append({
        'Step': step,
        'Removed': best_trial['removed'],
        'AIC': selected_model.aic,
        'Features': ', '.join(selected_features)
    })

history_df = pd.DataFrame(selection_history)
selected_coefficients = coefficient_table(selected_model)

print('Selected features:', selected_features)
display(history_df)
display(selected_coefficients.round(4))

print(f'Training R² = {selected_model.rsquared:.4f}')
print(f'Adjusted R² = {selected_model.rsquared_adj:.4f}')
print(f'AIC = {selected_model.aic:.2f}')

history_df.to_csv(
    'backward_AIC_history.csv',
    index=False,
    encoding='utf-8-sig'
)

pd.DataFrame(all_trials).to_csv(
    'backward_AIC_all_trials.csv',
    index=False,
    encoding='utf-8-sig'
)

selected_coefficients.to_csv(
    'selected_model_coefficients.csv',
    index=False,
    encoding='utf-8-sig'
)

#Apparent performance

In [ ]:
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

# 1. ใช้ข้อมูล training ที่เติม missing และทำ dummy แล้ว
# ต้องใช้ design matrix เดียวกับที่ fit โมเดล
X_train_final = make_design(selected_features)

y_observed = np.asarray(y, dtype=float)
y_predicted = np.asarray(
    selected_model.predict(X_train_final),
    dtype=float
)

# 2. Prediction performance
r2 = r2_score(y_observed, y_predicted)
rmse = np.sqrt(mean_squared_error(y_observed, y_predicted))
mae = mean_absolute_error(y_observed, y_predicted)

# Bias: predicted minus observed
bias = np.mean(y_predicted - y_observed)

# 3. Calibration: observed = intercept + slope × predicted
if np.std(y_predicted) > 1e-10:
    calibration_X = np.column_stack([
        np.ones(len(y_predicted)),
        y_predicted
    ])

    calibration_model = sm.OLS(
        y_observed,
        calibration_X
    ).fit()

    calibration_intercept, calibration_slope = (
        calibration_model.params
    )
else:
    calibration_intercept = np.nan
    calibration_slope = np.nan

# 4. สรุป apparent performance
apparent_performance = pd.DataFrame({
    'Metric': [
        'N records',
        'R²',
        'Adjusted R²',
        'RMSE (THB)',
        'MAE (THB)',
        'Prediction bias (THB)',
        'Calibration intercept (THB)',
        'Calibration slope',
        'Negative predictions'
    ],
    'Estimate': [
        len(y_observed),
        r2,
        selected_model.rsquared_adj,
        rmse,
        mae,
        bias,
        calibration_intercept,
        calibration_slope,
        int((y_predicted < 0).sum())
    ]
})

# ปรับค่าที่ใกล้ศูนย์จาก floating-point ให้แสดงเป็น 0
apparent_performance.loc[
    apparent_performance['Estimate'].abs() < 1e-8,
    'Estimate'
] = 0.0

print('Model:', selected_features)
print('Apparent performance — Training set')
display(apparent_performance.round(4))

# 5. บันทึกผล
apparent_performance.to_csv(
    'apparent_training_performance.csv',
    index=False,
    encoding='utf-8-sig'
)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

# =====================================================
# 1. Apparent predictions จาก training
# =====================================================
X_train_final = make_design(selected_features)

observed = np.asarray(y, dtype=float)
predicted = np.asarray(
    selected_model.predict(X_train_final),
    dtype=float
)

patient_ids = np.asarray(groups).astype(str)

if not (
    len(observed) == len(predicted) == len(patient_ids)
):
    raise ValueError('จำนวน outcome, prediction และ HN ไม่ตรงกัน')

if not (
    np.isfinite(observed).all()
    and np.isfinite(predicted).all()
):
    raise ValueError('พบ missing หรือ infinite values')

unique_patients = np.unique(patient_ids)

if len(unique_patients) < 2:
    raise ValueError('ต้องมีผู้ป่วยอย่างน้อย 2 คน')


# =====================================================
# 2. ฟังก์ชันคำนวณ performance
# =====================================================
def apparent_metrics(observed, predicted):

    results = {
        'R²': (
            r2_score(observed, predicted)
            if np.ptp(observed) > 0 else np.nan
        ),
        'RMSE (THB)': np.sqrt(
            mean_squared_error(observed, predicted)
        ),
        'MAE (THB)': mean_absolute_error(
            observed, predicted
        ),
        # ค่าลบ = ทำนายต่ำกว่าจริง
        'Prediction bias (THB)': np.mean(
            predicted - observed
        )
    }

    # Observed = intercept + slope × predicted
    if np.ptp(predicted) > 1e-8:

        calibration_X = np.column_stack([
            np.ones(len(predicted)),
            predicted
        ])

        intercept, slope = np.linalg.lstsq(
            calibration_X,
            observed,
            rcond=None
        )[0]

    else:
        intercept, slope = np.nan, np.nan

    results['Calibration intercept (THB)'] = intercept
    results['Calibration slope'] = slope

    return results


point_estimates = apparent_metrics(observed, predicted)


# =====================================================
# 3. Patient-cluster bootstrap
# สุ่มผู้ป่วยพร้อมรายการทั้งหมดของผู้ป่วยคนนั้น
# ไม่ refit prediction model
# =====================================================
rng = np.random.default_rng(42)
N_BOOTSTRAP = 1000

patient_indices = {
    patient: np.flatnonzero(patient_ids == patient)
    for patient in unique_patients
}

bootstrap_results = []

for _ in range(N_BOOTSTRAP):

    sampled_patients = rng.choice(
        unique_patients,
        size=len(unique_patients),
        replace=True
    )

    idx = np.concatenate([
        patient_indices[patient]
        for patient in sampled_patients
    ])

    bootstrap_results.append(
        apparent_metrics(
            observed[idx],
            predicted[idx]
        )
    )

bootstrap_df = pd.DataFrame(bootstrap_results)


# =====================================================
# 4. ตาราง Estimate และ percentile 95% CI
# =====================================================
apparent_performance = pd.DataFrame({
    'Metric': list(point_estimates.keys()),
    'Estimate': list(point_estimates.values()),
    'CI95_low': [
        bootstrap_df[metric].quantile(0.025)
        for metric in point_estimates
    ],
    'CI95_high': [
        bootstrap_df[metric].quantile(0.975)
        for metric in point_estimates
    ],
    'Valid_bootstrap_replicates': [
        bootstrap_df[metric].notna().sum()
        for metric in point_estimates
    ]
})

# ป้องกันการแสดง -0.0000 จาก floating-point
for col in ['Estimate', 'CI95_low', 'CI95_high']:
    apparent_performance.loc[
        apparent_performance[col].abs() < 1e-8,
        col
    ] = 0.0

print('Model:', selected_features)
print(
    f'Apparent: {len(observed)} records / '
    f'{len(unique_patients)} patients'
)

display(apparent_performance.round(4))

# Adjusted R² รายงานเป็น apparent point estimate แยกไว้
print(f'Adjusted R²: {selected_model.rsquared_adj:.4f}')
print(f'Negative predictions: {(predicted < 0).sum()}')


# =====================================================
# 5. บันทึกผล
# =====================================================
apparent_performance.to_csv(
    'apparent_performance_with_CI.csv',
    index=False,
    encoding='utf-8-sig'
)

# ดาวน์โหลดใน Colab หากต้องการ
# from google.colab import files
# files.download('apparent_performance_with_CI.csv')

##Gradio application for brain tumor cost prediction


In [ ]:
plt.figure(figsize=(7, 7))

plt.scatter(
    y_predicted,
    y_observed,
    alpha=0.4,
    s=25,
    label='Training records'
)

lower = min(0, y_observed.min(), y_predicted.min())
upper = max(y_observed.max(), y_predicted.max())

# เส้น ideal
plt.plot(
    [lower, upper],
    [lower, upper],
    'k--',
    label='Ideal calibration'
)

# เส้น calibration เฉพาะช่วงค่าทำนายที่มีข้อมูล
if np.isfinite(calibration_slope):
    grid = np.linspace(
        y_predicted.min(),
        y_predicted.max(),
        100
    )

    plt.plot(
        grid,
        calibration_intercept + calibration_slope * grid,
        color='red',
        linewidth=2,
        label='Linear calibration'
    )

plt.xlabel('Predicted total cost (THB)')
plt.ylabel('Observed total cost (THB)')
plt.title(f'Apparent calibration — Training set\nR² = {r2:.3f}')

plt.xlim(lower, upper)
plt.ylim(lower, upper)
plt.gca().set_aspect('equal', adjustable='box')
plt.legend()
plt.tight_layout()

plt.savefig(
    'apparent_calibration.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
from statsmodels.nonparametric.smoothers_lowess import lowess
from matplotlib.ticker import MultipleLocator

# =====================================================
# 1. Predictions จาก selected model บน training
# =====================================================
X_train_final = make_design(selected_features)

observed = np.asarray(y, dtype=float)
predicted = np.asarray(
    selected_model.predict(X_train_final),
    dtype=float
)

# groups ต้องเรียงตรงกับข้อมูล training ที่ใช้ fit
patient_ids = np.asarray(groups).astype(str)

if not (
    len(observed) == len(predicted) == len(patient_ids)
):
    raise ValueError('จำนวนข้อมูล outcome, prediction และ HN ไม่ตรงกัน')

if np.ptp(predicted) < 1e-8:
    raise ValueError('ค่าทำนายไม่มีความแปรปรวน จึงประมาณ calibration slope ไม่ได้')

r2 = r2_score(observed, predicted)

# Observed = intercept + slope × predicted
calibration_X = np.column_stack([
    np.ones(len(predicted)),
    predicted
])

intercept, slope = np.linalg.lstsq(
    calibration_X,
    observed,
    rcond=None
)[0]

# =====================================================
# 2. Patient-cluster bootstrap: 95% CI ของเส้น calibration
# คง fitted prediction เดิม ไม่ refit prediction model
# =====================================================
rng = np.random.default_rng(42)
n_bootstrap = 1000

unique_patients = np.unique(patient_ids)

patient_indices = {
    patient: np.flatnonzero(patient_ids == patient)
    for patient in unique_patients
}

# วาดเส้นเฉพาะช่วงค่าทำนายที่มีข้อมูล
grid = np.linspace(
    predicted.min(),
    predicted.max(),
    200
)

bootstrap_lines = []

for _ in range(n_bootstrap):

    sampled_patients = rng.choice(
        unique_patients,
        size=len(unique_patients),
        replace=True
    )

    idx = np.concatenate([
        patient_indices[patient]
        for patient in sampled_patients
    ])

    if np.ptp(predicted[idx]) < 1e-8:
        continue

    boot_intercept, boot_slope = np.linalg.lstsq(
        calibration_X[idx],
        observed[idx],
        rcond=None
    )[0]

    bootstrap_lines.append(
        boot_intercept + boot_slope * grid
    )

if len(bootstrap_lines) < 100:
    raise ValueError('จำนวน bootstrap ที่ประมาณ slope ได้มีน้อยเกินไป')

bootstrap_lines = np.asarray(bootstrap_lines)

ci_low, ci_high = np.percentile(
    bootstrap_lines,
    [2.5, 97.5],
    axis=0
)

# LOWESS แบบไม่ลดน้ำหนัก outlier
smooth = lowess(
    endog=observed,
    exog=predicted,
    frac=0.70,
    it=0,
    return_sorted=True
)

# =====================================================
# 3. วาดกราฟ หน่วยพันบาท
# =====================================================
scale = 1000.0

fig, ax = plt.subplots(figsize=(6.5, 6.5))

# จุดข้อมูล
ax.scatter(
    predicted / scale,
    observed / scale,
    s=20,
    color='#8CA6B8',
    alpha=0.32,
    label='Individual records',
    zorder=2
)

# กำหนดขอบเขตให้คล้ายภาพ โดยไม่ตัดข้อมูลที่อยู่นอกช่วง
plot_min = min(
    0.0,
    np.floor(min(predicted.min(), observed.min()) / scale / 250) * 250
)

plot_max = max(
    2050.0,
    np.ceil(max(predicted.max(), observed.max()) / scale / 250) * 250
)

# เส้น ideal
ax.plot(
    [plot_min, plot_max],
    [plot_min, plot_max],
    color='#222222',
    linestyle='--',
    linewidth=1.6,
    label='Ideal calibration',
    zorder=1
)

# แถบ 95% CI
ax.fill_between(
    grid / scale,
    ci_low / scale,
    ci_high / scale,
    color='#2C86BE',
    alpha=0.16,
    label='95% CI of linear calibration',
    zorder=1
)

# เส้น linear calibration
ax.plot(
    grid / scale,
    (intercept + slope * grid) / scale,
    color='#147EB3',
    linewidth=2.4,
    label='Linear calibration',
    zorder=3
)

# เส้น LOWESS
ax.plot(
    smooth[:, 0] / scale,
    smooth[:, 1] / scale,
    color='#C55A20',
    linewidth=2.5,
    label='LOWESS',
    zorder=4
)

# ข้อความมุมซ้ายบน คำนวณจากโมเดลจริง
intercept_display = 0.0 if abs(intercept) < 0.5 else intercept

ax.text(
    0.04, 0.96,
    f'R² = {r2:.3f}\n'
    f'Intercept = {intercept_display:,.0f} THB\n'
    f'Slope = {slope:.3f}',
    transform=ax.transAxes,
    ha='left',
    va='top',
    fontsize=11
)

ax.set_xlabel(
    'Predicted cost (thousand THB)',
    fontsize=12
)
ax.set_ylabel(
    'Observed cost (thousand THB)',
    fontsize=12
)

ax.set_xlim(plot_min, plot_max)
ax.set_ylim(plot_min, plot_max)
ax.set_aspect('equal', adjustable='box')

ax.xaxis.set_major_locator(MultipleLocator(500))
ax.yaxis.set_major_locator(MultipleLocator(250))

ax.grid(color='#EAEAEA', linewidth=0.8, alpha=0.7)
ax.set_axisbelow(True)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='both', labelsize=11)

# ภาพแนบไม่มี legend จึงปิดไว้
# หากต้องการเปิด:
# ax.legend(loc='lower right', fontsize=8, frameon=False)
plt.title(f'Apparent calibration — Training set')

plt.tight_layout()

plt.savefig(
    'apparent_calibration_styled.png',
    dpi=300,
    bbox_inches='tight'
)
plt.savefig(
    'apparent_calibration_styled.pdf',
    bbox_inches='tight'
)

# แสดงคำอธิบายเส้นและจุด
ax.legend(
    loc='lower right',
    fontsize=9,
    frameon=True,
    facecolor='white',
    edgecolor='none',
    framealpha=0.9
)

plt.tight_layout()

plt.savefig(
    'test_calibration_with_legend.png',
    dpi=300,
    bbox_inches='tight'
)

plt.savefig(
    'test_calibration_with_legend.pdf',
    bbox_inches='tight'
)
from matplotlib.ticker import MultipleLocator

ax.set_xlim(0, 1000)
ax.set_ylim(0, 1000)

# แสดง tick ทุก 250 พันบาท
ax.xaxis.set_major_locator(MultipleLocator(250))
ax.yaxis.set_major_locator(MultipleLocator(250))

ax.set_aspect('equal', adjustable='box')

plt.tight_layout()

plt.savefig(
    'apparent_calibration_0_1500.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

plt.show()

plt.show()

#Predictive performance of test df

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

# =====================================================
# 1. เตรียม test set
# =====================================================
test = test_df.copy().reset_index(drop=True)
test.columns = test.columns.str.strip()

if 'OP_TYPE' not in test.columns:
    test = test.rename(columns={'OP_WFME': 'OP_TYPE'})

# ใช้ mapping เดียวกับขั้นเตรียม training
# หากข้อมูลเป็น numeric อยู่แล้ว ไม่ต้องแก้ mapping
OP_MAPPING = globals().get('OP_TYPE_MAPPING', {})

binary_mapping_test = {
    'no': 0, 'yes': 1,
    '0': 0, '1': 1,
    '0.0': 0, '1.0': 1
}

sex_mapping_test = {
    'male': 0, 'female': 1,
    '1': 0, '2': 1,
    '1.0': 0, '2.0': 1
}


def numeric_test(series):
    raw = series.replace(r'^\s*$', np.nan, regex=True)
    result = pd.to_numeric(raw, errors='coerce')

    if (raw.notna() & result.isna()).any():
        raise ValueError(
            f'{series.name}: พบค่าที่แปลงเป็น numeric ไม่ได้'
        )

    return result.replace([np.inf, -np.inf], np.nan)


def mapped_test(series, mapping):
    text = (
        series.astype('string')
        .str.strip()
        .str.lower()
        .replace('', pd.NA)
    )

    result = text.map(mapping)

    if (text.notna() & result.isna()).any():
        raise ValueError(
            f'{series.name}: พบ label ที่ไม่อยู่ใน mapping'
        )

    return result.astype(float)


# เตรียมเฉพาะ predictors ที่ selected model ใช้
for feature in selected_features:

    if feature == 'sex_female':

        test[feature] = mapped_test(
            test['gender'],
            sex_mapping_test
        )

    elif feature in binary_features:

        test[feature] = mapped_test(
            test[feature],
            binary_mapping_test
        )

    elif feature == 'OP_TYPE':

        raw = (
            test[feature].astype('string')
            .str.strip()
            .replace('', pd.NA)
        )

        converted = pd.to_numeric(raw, errors='coerce')

        mapping = {
            str(k).strip().lower(): float(v)
            for k, v in OP_MAPPING.items()
        }

        converted = converted.fillna(
            raw.str.lower().map(mapping)
        )

        invalid = raw.notna() & converted.isna()

        if invalid.any():
            raise ValueError(
                'กรุณาใช้ OP_TYPE_MAPPING เดียวกับ training: '
                f'{raw[invalid].unique().tolist()}'
            )

        test[feature] = converted.astype(float)

    else:
        test[feature] = numeric_test(test[feature])

    # เติม missing ด้วยค่า training เท่านั้น
    test[feature] = (
        test[feature]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(imputation_values[feature])
    )

test['TOTAL_COST'] = numeric_test(test['TOTAL_COST'])
test['HN'] = test['HN'].astype('string').str.strip()

# =====================================================
# 2. ตรวจรายการที่ประเมินได้
# =====================================================
valid_outcome = (
    test['TOTAL_COST'].notna()
    & test['TOTAL_COST'].gt(0)
    & test['HN'].notna()
    & test['HN'].ne('')
).fillna(False)

supported = pd.Series(True, index=test.index)

for feature in selected_features:
    if feature in categorical_features:

        unknown = ~test[feature].isin(
            category_levels[feature]
        )

        print(
            f'{feature}: unseen categories = '
            f'{test.loc[unknown, feature].unique().tolist()}'
        )
        print(f'จำนวนรายการที่ไม่รองรับ: {unknown.sum()}')

        supported &= ~unknown

eligible = valid_outcome & supported

print(f'Test ทั้งหมด: {len(test)} รายการ')
print(f'ไม่มี outcome/HN ที่ใช้ได้: {(~valid_outcome).sum()} รายการ')
print(
    'มี outcome แต่ category ไม่รองรับ:',
    int((valid_outcome & ~supported).sum()),
    'รายการ'
)

test_eval = test.loc[eligible].copy()

if len(test_eval) < 3:
    raise ValueError('มีข้อมูล test ที่ประเมินได้ไม่เพียงพอ')

if test_eval['HN'].nunique() < 2:
    raise ValueError('ต้องมีผู้ป่วยอย่างน้อย 2 คนสำหรับ cluster bootstrap')

# =====================================================
# 3. สร้าง test design matrix ตาม training
# =====================================================
X_test_final = pd.DataFrame(
    {'const': 1.0},
    index=test_eval.index
)

for feature in selected_features:

    if feature in categorical_features:

        category = pd.Series(
            pd.Categorical(
                test_eval[feature],
                categories=category_levels[feature]
            ),
            index=test_eval.index
        )

        dummy = pd.get_dummies(
            category,
            prefix=feature,
            drop_first=True,
            dtype=float
        )

        X_test_final = pd.concat(
            [X_test_final, dummy],
            axis=1
        )

    else:
        X_test_final[feature] = test_eval[feature].astype(float)

# เรียง columns ให้ตรงกับโมเดลที่ฝึกแล้ว
model_columns = selected_model.model.exog_names

missing_design = set(model_columns) - set(X_test_final.columns)

if missing_design:
    raise ValueError(
        f'Test design ไม่ตรงกับ training: {missing_design}'
    )

X_test_final = X_test_final[model_columns].astype(float)

# =====================================================
# 4. ทำนายโดยใช้ selected_model เดิม
# =====================================================
y_test_observed = test_eval['TOTAL_COST'].to_numpy(dtype=float)

y_test_predicted = np.asarray(
    selected_model.predict(X_test_final),
    dtype=float
)

test_groups = test_eval['HN'].astype(str).to_numpy()


def calculate_performance(observed, predicted):
    result = {
        'R²': (
            r2_score(observed, predicted)
            if np.ptp(observed) > 0 else np.nan
        ),
        'RMSE (THB)': np.sqrt(
            mean_squared_error(observed, predicted)
        ),
        'MAE (THB)': mean_absolute_error(
            observed, predicted
        ),
        'Prediction bias (THB)': np.mean(
            predicted - observed
        )
    }

    # Calibration เป็นการประเมิน ไม่ได้นำไปแก้ prediction
    if np.ptp(predicted) > 1e-8:
        design = np.column_stack([
            np.ones(len(predicted)),
            predicted
        ])

        intercept, slope = np.linalg.lstsq(
            design, observed, rcond=None
        )[0]

    else:
        intercept, slope = np.nan, np.nan

    result['Calibration intercept (THB)'] = intercept
    result['Calibration slope'] = slope

    return result


point_estimates = calculate_performance(
    y_test_observed,
    y_test_predicted
)

# =====================================================
# 5. Patient-cluster bootstrap 95% CI
# =====================================================
rng = np.random.default_rng(42)
N_BOOTSTRAP = 1000

unique_patients = np.unique(test_groups)

patient_indices = {
    patient: np.flatnonzero(test_groups == patient)
    for patient in unique_patients
}

bootstrap_results = []

for _ in range(N_BOOTSTRAP):

    sampled = rng.choice(
        unique_patients,
        size=len(unique_patients),
        replace=True
    )

    idx = np.concatenate([
        patient_indices[patient]
        for patient in sampled
    ])

    bootstrap_results.append(
        calculate_performance(
            y_test_observed[idx],
            y_test_predicted[idx]
        )
    )

bootstrap_df = pd.DataFrame(bootstrap_results)

test_performance = pd.DataFrame({
    'Metric': list(point_estimates.keys()),
    'Estimate': list(point_estimates.values()),
    'CI95_low': [
        bootstrap_df[metric].quantile(0.025)
        for metric in point_estimates
    ],
    'CI95_high': [
        bootstrap_df[metric].quantile(0.975)
        for metric in point_estimates
    ],
    'Valid_bootstrap_replicates': [
        bootstrap_df[metric].notna().sum()
        for metric in point_estimates
    ]
})

print('Model:', selected_features)
print(
    f'Evaluated test: {len(test_eval)} records / '
    f'{test_eval["HN"].nunique()} patients'
)
print(
    'Negative predictions:',
    int((y_test_predicted < 0).sum())
)

display(test_performance.round(4))



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
from statsmodels.nonparametric.smoothers_lowess import lowess
from matplotlib.ticker import MultipleLocator

# =====================================================
# 1. ข้อมูล test ที่ผ่านการตรวจสอบแล้ว
# =====================================================
observed = np.asarray(y_test_observed, dtype=float)
predicted = np.asarray(y_test_predicted, dtype=float)
patient_ids = np.asarray(test_groups).astype(str)

if not (
    len(observed) == len(predicted) == len(patient_ids)
):
    raise ValueError('จำนวน outcome, prediction และ HN ไม่ตรงกัน')

if not (
    np.isfinite(observed).all()
    and np.isfinite(predicted).all()
):
    raise ValueError('พบ missing หรือ infinite values')

if np.ptp(predicted) < 1e-8:
    raise ValueError('ค่าทำนายไม่มีความแปรปรวน')

r2 = r2_score(observed, predicted)

# Calibration: observed = intercept + slope × predicted
# ใช้เพื่อประเมินเท่านั้น ไม่เปลี่ยนค่าทำนายเดิม
calibration_X = np.column_stack([
    np.ones(len(predicted)),
    predicted
])

intercept, slope = np.linalg.lstsq(
    calibration_X,
    observed,
    rcond=None
)[0]

# =====================================================
# 2. Bootstrap CI ของเส้น calibration ตามผู้ป่วย
# =====================================================
rng = np.random.default_rng(42)
n_bootstrap = 1000

unique_patients = np.unique(patient_ids)

if len(unique_patients) < 2:
    raise ValueError('ต้องมีผู้ป่วยอย่างน้อย 2 คน')

patient_indices = {
    patient: np.flatnonzero(patient_ids == patient)
    for patient in unique_patients
}

grid = np.linspace(
    predicted.min(),
    predicted.max(),
    200
)

bootstrap_coefficients = []

for _ in range(n_bootstrap):

    sampled_patients = rng.choice(
        unique_patients,
        size=len(unique_patients),
        replace=True
    )

    idx = np.concatenate([
        patient_indices[patient]
        for patient in sampled_patients
    ])

    if np.ptp(predicted[idx]) < 1e-8:
        continue

    coefficients = np.linalg.lstsq(
        calibration_X[idx],
        observed[idx],
        rcond=None
    )[0]

    bootstrap_coefficients.append(coefficients)

if len(bootstrap_coefficients) < 100:
    raise ValueError('จำนวน bootstrap ที่ใช้ได้มีน้อยเกินไป')

bootstrap_coefficients = np.asarray(bootstrap_coefficients)

bootstrap_lines = (
    bootstrap_coefficients[:, 0, None]
    + bootstrap_coefficients[:, 1, None] * grid
)

ci_low, ci_high = np.percentile(
    bootstrap_lines,
    [2.5, 97.5],
    axis=0
)

# =====================================================
# 3. LOWESS แบบ descriptive
# =====================================================
smooth = lowess(
    endog=observed,
    exog=predicted,
    frac=0.70,
    it=0,
    return_sorted=True
)

# =====================================================
# 4. วาดกราฟ หน่วยพันบาท
# =====================================================
scale = 1000.0

fig, ax = plt.subplots(figsize=(6.5, 6.5))

ax.scatter(
    predicted / scale,
    observed / scale,
    s=20,
    color='#8CA6B8',
    alpha=0.32,
    label='Test records',
    zorder=2
)

# ขอบเขตอย่างน้อย 0–2,050 พันบาท
# ขยายอัตโนมัติหากมีข้อมูลอยู่นอกช่วง
plot_min = min(
    0.0,
    np.floor(
        min(predicted.min(), observed.min()) / scale / 250
    ) * 250
)

plot_max = max(
    2050.0,
    np.ceil(
        max(predicted.max(), observed.max()) / scale / 250
    ) * 250
)

# เส้น ideal
ax.plot(
    [plot_min, plot_max],
    [plot_min, plot_max],
    color='#222222',
    linestyle='--',
    linewidth=1.6,
    label='Ideal calibration',
    zorder=1
)

# แถบ pointwise 95% CI
ax.fill_between(
    grid / scale,
    ci_low / scale,
    ci_high / scale,
    color='#2C86BE',
    alpha=0.16,
    label='95% CI of linear calibration',
    zorder=1
)

# เส้น calibration
ax.plot(
    grid / scale,
    (intercept + slope * grid) / scale,
    color='#147EB3',
    linewidth=2.4,
    label='Linear calibration',
    zorder=3
)

# เส้น LOWESS
ax.plot(
    smooth[:, 0] / scale,
    smooth[:, 1] / scale,
    color='#C55A20',
    linewidth=2.5,
    label='LOWESS',
    zorder=4
)

intercept_display = (
    0.0 if abs(intercept) < 0.5 else intercept
)

ax.text(
    0.04, 0.96,
    f'R² = {r2:.3f}\n'
    f'Intercept = {intercept_display:,.0f} THB\n'
    f'Slope = {slope:.3f}',
    transform=ax.transAxes,
    ha='left',
    va='top',
    fontsize=11
)

ax.set_title(
    f'Test calibration  ',
    #f'{len(unique_patients)} patients',
    fontsize=12
)

ax.set_xlabel('Predicted cost (thousand THB)', fontsize=12)
ax.set_ylabel('Observed cost (thousand THB)', fontsize=12)

ax.set_xlim(plot_min, plot_max)
ax.set_ylim(plot_min, plot_max)
ax.set_aspect('equal', adjustable='box')

ax.xaxis.set_major_locator(MultipleLocator(500))
ax.yaxis.set_major_locator(MultipleLocator(250))

ax.grid(color='#EAEAEA', linewidth=0.8, alpha=0.7)
ax.set_axisbelow(True)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='both', labelsize=11)

# เปิดบรรทัดนี้หากต้องการ legend
# ax.legend(loc='lower right', fontsize=8, frameon=False)

plt.tight_layout()

plt.savefig(
    'test_calibration_styled.png',
    dpi=300,
    bbox_inches='tight'
)

plt.savefig(
    'test_calibration_styled.pdf',
    bbox_inches='tight'
)

# แสดงคำอธิบายเส้นและจุด
ax.legend(
    loc='lower right',
    fontsize=9,
    frameon=True,
    facecolor='white',
    edgecolor='none',
    framealpha=0.9
)

plt.tight_layout()

plt.savefig(
    'test_calibration_with_legend.png',
    dpi=300,
    bbox_inches='tight'
)

plt.savefig(
    'test_calibration_with_legend.pdf',
    bbox_inches='tight'
)

from matplotlib.ticker import MultipleLocator

ax.set_xlim(0, 1000)
ax.set_ylim(0, 1000)

# แสดง tick ทุก 250 พันบาท
ax.xaxis.set_major_locator(MultipleLocator(250))
ax.yaxis.set_major_locator(MultipleLocator(250))

ax.set_aspect('equal', adjustable='box')

plt.tight_layout()

plt.savefig(
    'apparent_calibration_0_1500.png',
    dpi=300,
    bbox_inches='tight'
)

plt.show()

plt.show()

plt.show()

#OTHER ALGORITHM

In [ ]:
import numpy as np
import pandas as pd

from sklearn.compose import (
    ColumnTransformer,
    TransformedTargetRegressor
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import GroupKFold, GridSearchCV

from sklearn.linear_model import (
    LinearRegression,
    ElasticNet,
    GammaRegressor
)
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

from catboost import CatBoostRegressor

FEATURES = [
    'age', 'Pre_KPS',
    'DYSLIPID', 'LUNG', 'PRE_SEIZURE',
    'OP_TYPE'
]

NUMERIC = ['age', 'Pre_KPS']
BINARY = ['DYSLIPID', 'LUNG', 'PRE_SEIZURE']
CATEGORICAL = ['OP_TYPE']

TARGET = 'TOTAL_COST'
SEED = 42

In [ ]:
def prepare_data(df):
    data = df.copy()
    data.columns = data.columns.str.strip()

    if 'OP_TYPE' not in data.columns:
        data = data.rename(columns={'OP_WFME': 'OP_TYPE'})

    required = FEATURES + [TARGET, 'HN']
    missing = [col for col in required if col not in data.columns]

    if missing:
        raise ValueError(f'ไม่พบคอลัมน์: {missing}')

    # เก็บตำแหน่งรายการก่อนกรอง
    data['_input_row'] = np.arange(len(data))

    for col in NUMERIC + [TARGET]:
        raw = data[col].replace(r'^\s*$', np.nan, regex=True)
        converted = pd.to_numeric(raw, errors='coerce')

        if (raw.notna() & converted.isna()).any():
            raise ValueError(f'{col}: พบค่าที่ไม่ใช่ตัวเลข')

        data[col] = converted.replace(
            [np.inf, -np.inf], np.nan
        )

    binary_mapping = {
        'yes': 1.0, 'no': 0.0,
        '1': 1.0, '0': 0.0,
        '1.0': 1.0, '0.0': 0.0
    }

    for col in BINARY:
        text = (
            data[col].astype('string')
            .str.strip().str.lower()
            .replace('', pd.NA)
        )

        converted = text.map(binary_mapping)
        invalid = text.notna() & converted.isna()

        if invalid.any():
            raise ValueError(
                f'{col}: ไม่รู้จักค่า {text[invalid].unique().tolist()}'
            )

        data[col] = converted.astype(float)

    # ทำให้รหัส 1 และ 1.0 เป็น category เดียวกัน
    def normalize_operation(value):
        if pd.isna(value) or str(value).strip() == '':
            return np.nan

        text = str(value).strip()

        try:
            number = float(text)

            if np.isfinite(number):
                return (
                    str(int(number))
                    if number.is_integer()
                    else str(number)
                )

        except ValueError:
            pass

        return text

    data['OP_TYPE'] = (
        data['OP_TYPE'].map(normalize_operation).astype(object)
    )

    data['HN'] = data['HN'].astype('string').str.strip()

    # ใช้เกณฑ์เดียวกันทุกโมเดล รวม Gamma ที่ต้องการ outcome > 0
    valid = (
        data[TARGET].notna()
        & data[TARGET].gt(0)
        & data['HN'].notna()
        & data['HN'].ne('')
    ).fillna(False)

    print(f'ตัด outcome/HN ที่ใช้ไม่ได้: {(~valid).sum()} รายการ')

    return data.loc[valid].reset_index(drop=True)


train_ml = prepare_data(train_df)
test_ml = prepare_data(test_df)

overlap = set(train_ml['HN']) & set(test_ml['HN'])

if overlap:
    raise ValueError('พบผู้ป่วยซ้ำระหว่าง training และ test')

for col in FEATURES:
    if train_ml[col].isna().all():
        raise ValueError(f'{col}: ไม่มีข้อมูลใน training')

In [ ]:
known_operations = set(train_ml['OP_TYPE'].dropna())

supported = (
    test_ml['OP_TYPE'].isna()
    | test_ml['OP_TYPE'].isin(known_operations)
)

excluded_test = test_ml.loc[
    ~supported,
    ['_input_row', 'OP_TYPE']
].copy()

print('รายการที่ชนิดผ่าตัดไม่พบใน training:')
display(excluded_test)

test_ml = test_ml.loc[supported].reset_index(drop=True)

if train_ml['HN'].nunique() < 5:
    raise ValueError('ต้องมีผู้ป่วย training อย่างน้อย 5 คน')

if test_ml['HN'].nunique() < 2:
    raise ValueError('จำนวนผู้ป่วย test ไม่เพียงพอ')

X_train = train_ml[FEATURES].copy()
X_test = test_ml[FEATURES].copy()

y_train = train_ml[TARGET].to_numpy(dtype=float)
y_test = test_ml[TARGET].to_numpy(dtype=float)

groups_train = train_ml['HN'].astype(str).to_numpy()
groups_test = test_ml['HN'].astype(str).to_numpy()

print(f'Train: {len(X_train)} records')
print(f'Test: {len(X_test)} records')
print('Predictors:', FEATURES)

In [ ]:
def make_preprocessor():
    return ColumnTransformer(
        transformers=[
            (
                'numeric',
                Pipeline([
                    ('imputer', SimpleImputer(strategy='median')),
                    ('scaler', StandardScaler())
                ]),
                NUMERIC
            ),
            (
                'binary',
                SimpleImputer(strategy='most_frequent'),
                BINARY
            ),
            (
                'operation',
                Pipeline([
                    (
                        'imputer',
                        SimpleImputer(strategy='most_frequent')
                    ),
                    (
                        'encoder',
                        OneHotEncoder(
                            drop='first',
                            handle_unknown='ignore',
                            sparse_output=False
                        )
                    )
                ]),
                CATEGORICAL
            )
        ],
        sparse_threshold=0
    )


def make_pipeline(estimator):
    return Pipeline([
        ('preprocess', make_preprocessor()),
        ('model', estimator)
    ])


# ใช้ folds เดียวกันทุกอัลกอริทึม
cv_splits = list(
    GroupKFold(n_splits=5).split(
        X_train,
        y_train,
        groups=groups_train
    )
)

# ตรวจสอบผู้ป่วยไม่ข้าม fold
for training_idx, validation_idx in cv_splits:
    assert set(groups_train[training_idx]).isdisjoint(
        groups_train[validation_idx]
    )

fitted_models = {}
tuning_results = []


def tune_model(name, estimator, parameter_grid):
    search = GridSearchCV(
        estimator=make_pipeline(estimator),
        param_grid=parameter_grid,
        scoring='neg_root_mean_squared_error',
        cv=cv_splits,
        refit=True,
        n_jobs=2,
        error_score='raise'
    )

    # ใช้ training เท่านั้น
    search.fit(X_train, y_train)

    fitted_models[name] = search.best_estimator_

    tuning_results.append({
        'Model': name,
        'Best_CV_RMSE': -search.best_score_,
        'Best_parameters': str(search.best_params_)
    })

    print(name)
    print('Best parameters:', search.best_params_)
    print(f'CV RMSE: {-search.best_score_:,.2f} THB')

    return search

# OLS

In [ ]:
ols_search = tune_model(
    name='OLS',
    estimator=LinearRegression(),
    parameter_grid={}
)

#Elastic Net

In [ ]:
elastic_net = TransformedTargetRegressor(
    regressor=ElasticNet(
        max_iter=50000,
        tol=1e-5,
        random_state=SEED
    ),
    transformer=StandardScaler()
)

elastic_search = tune_model(
    name='ElasticNet',
    estimator=elastic_net,
    parameter_grid={
        'model__regressor__alpha': [
            0.0001, 0.001, 0.01, 0.1, 1.0
        ],
        'model__regressor__l1_ratio': [
            0.1, 0.5, 0.9
        ]
    }
)

#Random forest

In [ ]:
rf_search = tune_model(
    name='RandomForest',
    estimator=RandomForestRegressor(
        n_estimators=300,
        random_state=SEED,
        n_jobs=1
    ),
    parameter_grid={
        'model__max_depth': [4, 8, None],
        'model__min_samples_leaf': [5, 15],
        'model__max_features': [0.7, 1.0]
    }
)

#CatBoost

In [ ]:
catboost_search = tune_model(
    name='CatBoost',
    estimator=CatBoostRegressor(
        loss_function='RMSE',
        random_seed=SEED,
        verbose=False,
        allow_writing_files=False,
        thread_count=1
    ),
    parameter_grid={
        'model__depth': [3, 5],
        'model__iterations': [300, 600],
        'model__learning_rate': [0.03],
        'model__l2_leaf_reg': [3.0, 10.0]
    }
)

#Gramma

In [ ]:
gamma_search = tune_model(
    name='Gamma',
    estimator=GammaRegressor(
        max_iter=10000,
        tol=1e-7
    ),
    parameter_grid={
        'model__alpha': [
            0.0, 0.001, 0.01, 0.1, 1.0, 10.0
        ]
    }
)

display(pd.DataFrame(tuning_results))

#apparent and test with  95% CI

In [ ]:
def calculate_metrics(observed, predicted):

    observed = np.asarray(observed, dtype=float)
    predicted = np.asarray(predicted, dtype=float)

    if not np.isfinite(predicted).all():
        raise ValueError('พบ prediction ที่ไม่เป็น finite')

    if np.ptp(predicted) > 1e-8:
        calibration_X = np.column_stack([
            np.ones(len(predicted)),
            predicted
        ])

        intercept, slope = np.linalg.lstsq(
            calibration_X,
            observed,
            rcond=None
        )[0]
    else:
        intercept, slope = np.nan, np.nan

    return {
        'R2': (
            r2_score(observed, predicted)
            if np.ptp(observed) > 0 else np.nan
        ),
        'RMSE_THB': np.sqrt(
            mean_squared_error(observed, predicted)
        ),
        'MAE_THB': mean_absolute_error(
            observed, predicted
        ),
        'Bias_THB': np.mean(predicted - observed),
        'Calibration_intercept_THB': intercept,
        'Calibration_slope': slope
    }


def performance_with_ci(observed, predicted, patient_ids, B=1000):

    point = calculate_metrics(observed, predicted)

    rng = np.random.default_rng(SEED)
    unique_patients = np.unique(patient_ids)

    patient_indices = {
        patient: np.flatnonzero(patient_ids == patient)
        for patient in unique_patients
    }

    bootstrap_results = []

    for _ in range(B):
        sampled_patients = rng.choice(
            unique_patients,
            size=len(unique_patients),
            replace=True
        )

        idx = np.concatenate([
            patient_indices[patient]
            for patient in sampled_patients
        ])

        bootstrap_results.append(
            calculate_metrics(
                observed[idx],
                predicted[idx]
            )
        )

    bootstrap_df = pd.DataFrame(bootstrap_results)

    return pd.DataFrame({
        'Metric': list(point.keys()),
        'Estimate': list(point.values()),
        'CI95_low': [
            bootstrap_df[k].quantile(0.025)
            for k in point
        ],
        'CI95_high': [
            bootstrap_df[k].quantile(0.975)
            for k in point
        ],
        'Valid_bootstrap_replicates': [
            bootstrap_df[k].notna().sum()
            for k in point
        ]
    })


all_results = []

train_predictions = pd.DataFrame({
    'Input_row': train_ml['_input_row'],
    'Observed_cost': y_train
})

test_predictions = pd.DataFrame({
    'Input_row': test_ml['_input_row'],
    'Observed_cost': y_test
})

for name, model in fitted_models.items():

    # ไม่มีการ fit ใหม่ในขั้นประเมิน
    pred_train = np.asarray(model.predict(X_train))
    pred_test = np.asarray(model.predict(X_test))

    train_predictions[name] = pred_train
    test_predictions[name] = pred_test

    for dataset, observed, predicted, patient_ids in [
        ('Apparent', y_train, pred_train, groups_train),
        ('Test', y_test, pred_test, groups_test)
    ]:

        table = performance_with_ci(
            observed,
            predicted,
            patient_ids,
            B=1000
        )

        table.insert(0, 'Dataset', dataset)
        table.insert(0, 'Model', name)

        table['N_records'] = len(observed)
        table['N_patients'] = len(np.unique(patient_ids))
        table['Negative_predictions'] = int(
            (predicted < 0).sum()
        )

        all_results.append(table)

performance = pd.concat(all_results, ignore_index=True)

In [ ]:
report = performance.copy()

def format_result(row):
    digits = 3 if row['Metric'] in [
        'R2', 'Calibration_slope'
    ] else 0

    return (
        f"{row['Estimate']:,.{digits}f} "
        f"({row['CI95_low']:,.{digits}f}–"
        f"{row['CI95_high']:,.{digits}f})"
    )

report['Estimate (95% CI)'] = report.apply(
    format_result,
    axis=1
)

for dataset in ['Apparent', 'Test']:

    print(f'\n{dataset} performance')

    display(
        report.loc[report['Dataset'] == dataset].pivot(
            index='Model',
            columns='Metric',
            values='Estimate (95% CI)'
        )
    )

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.metrics import r2_score
from statsmodels.nonparametric.smoothers_lowess import lowess
from matplotlib.ticker import MultipleLocator

# =====================================================
# 1. เตรียมข้อมูลและ bootstrap indices ชุดเดียวกันทุกโมเดล
# =====================================================
observed = test_predictions['Observed_cost'].to_numpy(dtype=float)
patient_ids = np.asarray(groups_test).astype(str)

model_names = [
    'OLS',
    'ElasticNet',
    'RandomForest',
    'CatBoost',
    'Gamma'
]

missing_models = [
    name for name in model_names
    if name not in test_predictions.columns
]

if missing_models:
    raise ValueError(f'ไม่พบ predictions ของ: {missing_models}')

if len(observed) != len(patient_ids):
    raise ValueError('จำนวน predictions และ groups_test ไม่ตรงกัน')

if not np.isfinite(observed).all():
    raise ValueError('พบ missing หรือ infinite observed cost')

unique_patients = np.unique(patient_ids)

if len(unique_patients) < 2:
    raise ValueError('ต้องมีผู้ป่วย test อย่างน้อย 2 คน')

rng = np.random.default_rng(42)
N_BOOTSTRAP = 1000

patient_indices = {
    patient: np.flatnonzero(patient_ids == patient)
    for patient in unique_patients
}

# คงลำดับข้อมูลจาก Step 10 ห้าม sort predictions แยกจาก groups_test
bootstrap_indices = []

for _ in range(N_BOOTSTRAP):

    sampled_patients = rng.choice(
        unique_patients,
        size=len(unique_patients),
        replace=True
    )

    bootstrap_indices.append(
        np.concatenate([
            patient_indices[patient]
            for patient in sampled_patients
        ])
    )

output_dir = Path('test_calibration_models')
output_dir.mkdir(exist_ok=True)


# =====================================================
# 2. คำนวณ calibration ของแต่ละโมเดล
# =====================================================
calibration_results = {}

for name in model_names:

    predicted = test_predictions[name].to_numpy(dtype=float)

    if not np.isfinite(predicted).all():
        raise ValueError(f'{name}: พบ prediction ที่ไม่เป็น finite')

    if np.ptp(predicted) < 1e-8:
        raise ValueError(
            f'{name}: predictions ไม่มีความแปรปรวน '
            'จึงประมาณ calibration slope ไม่ได้'
        )

    calibration_X = np.column_stack([
        np.ones(len(predicted)),
        predicted
    ])

    # ประเมิน calibration เท่านั้น ไม่แก้ค่าทำนายเดิม
    intercept, slope = np.linalg.lstsq(
        calibration_X,
        observed,
        rcond=None
    )[0]

    grid = np.linspace(
        predicted.min(),
        predicted.max(),
        200
    )

    bootstrap_lines = []

    for idx in bootstrap_indices:

        if np.ptp(predicted[idx]) < 1e-8:
            continue

        boot_intercept, boot_slope = np.linalg.lstsq(
            calibration_X[idx],
            observed[idx],
            rcond=None
        )[0]

        bootstrap_lines.append(
            boot_intercept + boot_slope * grid
        )

    if len(bootstrap_lines) < 100:
        raise ValueError(
            f'{name}: จำนวน bootstrap ที่ใช้ได้ไม่เพียงพอ'
        )

    ci_low, ci_high = np.percentile(
        np.asarray(bootstrap_lines),
        [2.5, 97.5],
        axis=0
    )

    smooth = lowess(
        endog=observed,
        exog=predicted,
        frac=0.70,
        it=0,
        return_sorted=True
    )

    calibration_results[name] = {
        'predicted': predicted,
        'r2': r2_score(observed, predicted),
        'intercept': intercept,
        'slope': slope,
        'grid': grid,
        'ci_low': ci_low,
        'ci_high': ci_high,
        'smooth': smooth
    }


# =====================================================
# 3. ฟังก์ชันวาดกราฟ
# =====================================================
def draw_calibration(ax, name):

    result = calibration_results[name]
    predicted = result['predicted']
    grid = result['grid']
    smooth = result['smooth']

    # หน่วยพันบาท
    scale = 1000.0

    ax.scatter(
        predicted / scale,
        observed / scale,
        s=20,
        color='#8CA6B8',
        alpha=0.32,
        label='Test records',
        zorder=2
    )

    ax.plot(
        [0, 1500],
        [0, 1500],
        color='#222222',
        linestyle='--',
        linewidth=1.6,
        label='Ideal calibration',
        zorder=1
    )

    ax.fill_between(
        grid / scale,
        result['ci_low'] / scale,
        result['ci_high'] / scale,
        color='#2C86BE',
        alpha=0.16,
        label='95% CI of linear calibration',
        zorder=1
    )

    ax.plot(
        grid / scale,
        (
            result['intercept']
            + result['slope'] * grid
        ) / scale,
        color='#147EB3',
        linewidth=2.4,
        label='Linear calibration',
        zorder=3
    )

    ax.plot(
        smooth[:, 0] / scale,
        smooth[:, 1] / scale,
        color='#C55A20',
        linewidth=2.5,
        label='LOWESS',
        zorder=4
    )

    intercept_display = (
        0.0
        if abs(result['intercept']) < 0.5
        else result['intercept']
    )

    ax.text(
        0.04, 0.96,
        f"R² = {result['r2']:.3f}\n"
        f"Intercept = {intercept_display:,.0f} THB\n"
        f"Slope = {result['slope']:.3f}",
        transform=ax.transAxes,
        ha='left',
        va='top',
        fontsize=10
    )

    ax.set_title(name, fontsize=12)
    ax.set_xlabel('Predicted cost (thousand THB)')
    ax.set_ylabel('Observed cost (thousand THB)')

    ax.set_xlim(0, 1500)
    ax.set_ylim(0, 1500)
    ax.set_aspect('equal', adjustable='box')

    ax.xaxis.set_major_locator(MultipleLocator(250))
    ax.yaxis.set_major_locator(MultipleLocator(250))

    ax.grid(color='#EAEAEA', linewidth=0.8, alpha=0.7)
    ax.set_axisbelow(True)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)


# =====================================================
# 4. รูปรวมทุกโมเดล
# =====================================================
fig, axes = plt.subplots(2, 3, figsize=(17, 11))
axes = axes.ravel()

for ax, name in zip(axes, model_names):
    draw_calibration(ax, name)

# ช่องสุดท้ายสำหรับ legend
axes[-1].axis('off')

handles, labels = axes[0].get_legend_handles_labels()

axes[-1].legend(
    handles,
    labels,
    loc='center',
    frameon=False,
    fontsize=11
)

fig.suptitle(
    f'Test calibration: {len(observed)} records / '
    f'{len(unique_patients)} patients',
    fontsize=15
)

fig.tight_layout(rect=[0, 0, 1, 0.96])

fig.savefig(
    output_dir / 'test_calibration_all_models.png',
    dpi=300,
    bbox_inches='tight'
)

fig.savefig(
    output_dir / 'test_calibration_all_models.pdf',
    bbox_inches='tight'
)

plt.show()


# =====================================================
# 5. แสดงและบันทึกรูปแยกแต่ละโมเดล
# =====================================================
for name in model_names:

    fig, ax = plt.subplots(figsize=(6.5, 6.5))
    draw_calibration(ax, name)

    ax.legend(
        loc='lower right',
        fontsize=8,
        frameon=True,
        facecolor='white',
        edgecolor='none',
        framealpha=0.9
    )

    fig.tight_layout()

    fig.savefig(
        output_dir / f'test_calibration_{name}.png',
        dpi=300,
        bbox_inches='tight'
    )

    fig.savefig(
        output_dir / f'test_calibration_{name}.pdf',
        bbox_inches='tight'
    )

    plt.show()

#bootstrap

In [ ]:
required = [OBSERVED_COLUMN] + MODEL_NAMES

missing_columns = [
    col for col in required if col not in df.columns
]

if missing_columns:
    raise ValueError(f'ไม่พบคอลัมน์: {missing_columns}')

for col in required:
    raw = df[col].replace(r'^\s*$', np.nan, regex=True)
    converted = pd.to_numeric(raw, errors='coerce')

    if (raw.notna() & converted.isna()).any():
        raise ValueError(f'{col}: พบค่าที่ไม่ใช่ตัวเลข')

    df[col] = converted.replace([np.inf, -np.inf], np.nan)

# ใช้ complete rows ชุดเดียวกันทุกโมเดล
valid = df[required].notna().all(axis=1)

print(f'แถวที่ตัดเพราะ outcome/prediction ไม่ครบ: {(~valid).sum()}')

evaluation_df = df.loc[valid].copy().reset_index(drop=True)

if len(evaluation_df) < 3:
    raise ValueError('จำนวนข้อมูลไม่เพียงพอ')

observed = evaluation_df[OBSERVED_COLUMN].to_numpy(dtype=float)

if np.ptp(observed) == 0:
    raise ValueError('Observed cost ไม่มีความแปรปรวน จึงประเมิน R² ไม่ได้')

N_BOOTSTRAP = 1000
rng = np.random.default_rng(42)

if 'HN' in evaluation_df.columns:
    hn = (
        evaluation_df['HN']
        .astype('string')
        .str.strip()
    )

    if hn.isna().any() or hn.eq('').any():
        raise ValueError('มีคอลัมน์ HN แต่บางรายการไม่มีรหัสผู้ป่วย')

    group_ids = hn.to_numpy()
    BOOTSTRAP_METHOD = 'Patient-cluster bootstrap'

else:
    # แต่ละแถวเป็นหน่วยสุ่ม ไม่ได้อ้างว่าเป็นผู้ป่วยไม่ซ้ำ
    group_ids = np.arange(len(evaluation_df))
    BOOTSTRAP_METHOD = 'Row bootstrap'

unique_groups = np.unique(group_ids)

if len(unique_groups) < 2:
    raise ValueError('จำนวนหน่วย bootstrap ไม่เพียงพอ')

group_indices = {
    group: np.flatnonzero(group_ids == group)
    for group in unique_groups
}

# ใช้ชุดสุ่มเดียวกันทุกโมเดล
bootstrap_indices = []

for _ in range(N_BOOTSTRAP):
    sampled_groups = rng.choice(
        unique_groups,
        size=len(unique_groups),
        replace=True
    )

    idx = np.concatenate([
        group_indices[group]
        for group in sampled_groups
    ])

    bootstrap_indices.append(idx)

print(f'Evaluated records: {len(evaluation_df)}')
print('Bootstrap method:', BOOTSTRAP_METHOD)

In [ ]:
def calculate_metrics(y, p):
    if np.ptp(p) > 1e-8:
        design = np.column_stack([
            np.ones(len(p)),
            p
        ])

        intercept, slope = np.linalg.lstsq(
            design, y, rcond=None
        )[0]
    else:
        intercept, slope = np.nan, np.nan

    return {
        'R2': (
            r2_score(y, p)
            if np.ptp(y) > 0 else np.nan
        ),
        'RMSE_THB': np.sqrt(mean_squared_error(y, p)),
        'MAE_THB': mean_absolute_error(y, p),
        'Bias_THB': np.mean(p - y),
        'Calibration_intercept_THB': intercept,
        'Calibration_slope': slope
    }


results = []
bootstrap_by_model = {}
point_by_model = {}

for name in MODEL_NAMES:
    predicted = evaluation_df[name].to_numpy(dtype=float)

    point = calculate_metrics(observed, predicted)
    point_by_model[name] = point

    bootstrap_values = [
        calculate_metrics(
            observed[idx],
            predicted[idx]
        )
        for idx in bootstrap_indices
    ]

    boot = pd.DataFrame(bootstrap_values)
    bootstrap_by_model[name] = boot

    for metric, estimate in point.items():
        results.append({
            'Model': name,
            'Dataset': DATASET_LABEL,
            'Metric': metric,
            'Estimate': estimate,
            'CI95_low': boot[metric].quantile(0.025),
            'CI95_high': boot[metric].quantile(0.975),
            'Valid_bootstrap_replicates':
                boot[metric].notna().sum(),
            'N_records': len(observed),
            'Negative_predictions': int((predicted < 0).sum()),
            'Bootstrap_method': BOOTSTRAP_METHOD
        })

performance = pd.DataFrame(results)

report = performance.copy()

def format_result(row):
    digits = 3 if row['Metric'] in [
        'R2', 'Calibration_slope'
    ] else 0

    return (
        f"{row['Estimate']:,.{digits}f} "
        f"({row['CI95_low']:,.{digits}f}–"
        f"{row['CI95_high']:,.{digits}f})"
    )

report['Estimate (95% CI)'] = report.apply(
    format_result, axis=1
)

summary_table = report.pivot(
    index='Model',
    columns='Metric',
    values='Estimate (95% CI)'
).reindex(MODEL_NAMES)

display(summary_table)

performance.to_csv(
    OUTPUT_DIR / 'performance_with_CI.csv',
    index=False,
    encoding='utf-8-sig'
)

summary_table.to_csv(
    OUTPUT_DIR / 'performance_formatted.csv',
    encoding='utf-8-sig'
)

In [ ]:
def draw_calibration(ax, name):
    predicted = evaluation_df[name].to_numpy(dtype=float)
    point = point_by_model[name]
    boot = bootstrap_by_model[name]

    scale = 1000.0

    ax.scatter(
        predicted / scale,
        observed / scale,
        s=20,
        color='#8CA6B8',
        alpha=0.32,
        label='Individual records',
        zorder=2
    )

    ax.plot(
        [0, 1500], [0, 1500],
        color='#222222',
        linestyle='--',
        linewidth=1.6,
        label='Ideal calibration',
        zorder=1
    )

    if np.isfinite(point['Calibration_slope']):

        grid = np.linspace(
            predicted.min(),
            predicted.max(),
            200
        )

        valid_boot = boot[
            [
                'Calibration_intercept_THB',
                'Calibration_slope'
            ]
        ].dropna()

        if len(valid_boot) >= 100:
            lines = (
                valid_boot['Calibration_intercept_THB']
                .to_numpy()[:, None]
                +
                valid_boot['Calibration_slope']
                .to_numpy()[:, None] * grid
            )

            lower, upper = np.percentile(
                lines, [2.5, 97.5], axis=0
            )

            ax.fill_between(
                grid / scale,
                lower / scale,
                upper / scale,
                color='#2C86BE',
                alpha=0.16,
                label='95% CI of linear calibration',
                zorder=1
            )

        ax.plot(
            grid / scale,
            (
                point['Calibration_intercept_THB']
                + point['Calibration_slope'] * grid
            ) / scale,
            color='#147EB3',
            linewidth=2.4,
            label='Linear calibration',
            zorder=3
        )

        smooth = lowess(
            observed,
            predicted,
            frac=0.70,
            it=0,
            return_sorted=True
        )

        ax.plot(
            smooth[:, 0] / scale,
            smooth[:, 1] / scale,
            color='#C55A20',
            linewidth=2.5,
            label='LOWESS',
            zorder=4
        )

    intercept = point['Calibration_intercept_THB']

    if np.isfinite(intercept) and abs(intercept) < 0.5:
        intercept = 0.0

    ax.text(
        0.04, 0.96,
        f"R² = {point['R2']:.3f}\n"
        f"Intercept = {intercept:,.0f} THB\n"
        f"Slope = {point['Calibration_slope']:.3f}",
        transform=ax.transAxes,
        ha='left',
        va='top',
        fontsize=10
    )

    ax.set_title(name, fontsize=12)
    ax.set_xlabel('Predicted cost (thousand THB)')
    ax.set_ylabel('Observed cost (thousand THB)')

    ax.set_xlim(0, 750)
    ax.set_ylim(0, 750)
    ax.set_aspect('equal', adjustable='box')

    ax.xaxis.set_major_locator(MultipleLocator(250))
    ax.yaxis.set_major_locator(MultipleLocator(250))

    ax.grid(color='#EAEAEA', linewidth=0.8, alpha=0.7)
    ax.set_axisbelow(True)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)


# รูปแยกแต่ละโมเดล
for name in MODEL_NAMES:
    fig, ax = plt.subplots(figsize=(6.5, 6.5))

    draw_calibration(ax, name)

    ax.legend(
        loc='lower right',
        fontsize=8,
        frameon=True,
        facecolor='white',
        edgecolor='none',
        framealpha=0.9
    )

    fig.tight_layout()

    for extension in ['png', 'pdf']:
        fig.savefig(
            OUTPUT_DIR / f'calibration_{name}.{extension}',
            dpi=300,
            bbox_inches='tight'
        )

    plt.show()


# รูปรวม
fig, axes = plt.subplots(2, 3, figsize=(17, 11))
axes = axes.ravel()

for ax, name in zip(axes, MODEL_NAMES):
    draw_calibration(ax, name)

axes[-1].axis('off')

handles, labels = axes[0].get_legend_handles_labels()
axes[-1].legend(
    handles, labels,
    loc='center',
    frameon=False,
    fontsize=11
)

fig.suptitle(
    f'{DATASET_LABEL}: {len(observed)} records',
    fontsize=15
)

fig.tight_layout(rect=[0, 0, 1, 0.96])

for extension in ['png', 'pdf']:
    fig.savefig(
        OUTPUT_DIR / f'calibration_all_models.{extension}',
        dpi=300,
        bbox_inches='tight'
    )

plt.show()

## Launch Gradio
Running this cell creates a temporary public share link. The app uses only the fitted JSON model and input values; it does not expose source CSVs. Do not enter patient identifiers. Unseen procedure categories are rejected. For preoperative use, OP_WFME must describe the planned procedure. Stop the cell/runtime to stop the app.

#Call prediction model

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import joblib

BASE_DIR = Path(
    '/content/drive/MyDrive/'
    'Cost of neurosurgical operation/saved_models'
)

# ค้นหาไฟล์โมเดลในโฟลเดอร์ที่ตั้งชื่อตามเวลา
model_files = sorted(
    BASE_DIR.glob('*/all_fitted_models.joblib'),
    key=lambda path: path.parent.name
)

if not model_files:
    raise FileNotFoundError(
        f'ไม่พบ all_fitted_models.joblib ใน {BASE_DIR}'
    )

# เลือกโฟลเดอร์ล่าสุด
MODEL_PATH = model_files[-1]
MODEL_DIR = MODEL_PATH.parent

print('กำลังใช้โมเดลจาก:', MODEL_DIR)

# ดู versions ที่ใช้บันทึกโมเดล
metadata_path = MODEL_DIR / 'metadata.json'

if metadata_path.exists():
    with open(metadata_path, encoding='utf-8') as f:
        metadata = json.load(f)

    print('Package versions ที่ใช้ฝึก:')
    print(metadata['versions'])

# โหลดทุกโมเดลพร้อม preprocessing pipelines
fitted_models = joblib.load(MODEL_PATH)

print('โหลดสำเร็จ:', list(fitted_models.keys()))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
กำลังใช้โมเดลจาก: /content/drive/MyDrive/Cost of neurosurgical operation/saved_models/20260914_041307_006424
Package versions ที่ใช้ฝึก:
{'python': '3.13.15', 'numpy': '2.1.3', 'pandas': '2.2.3', 'scikit-learn': '1.6.1', 'catboost': '1.2.10', 'joblib': '1.6.0'}
โหลดสำเร็จ: ['OLS', 'ElasticNet', 'RandomForest', 'CatBoost', 'Gamma']


#Step 1 - Install Gradio

In [ ]:
%pip -q install "gradio>=5,<7"

#Step 2 - Application

In [ ]:
import numpy as np
import pandas as pd
import gradio as gr

MODEL_NAMES = [
    'OLS',
    'ElasticNet',
    'RandomForest',
    'CatBoost',
    'Gamma'
]

APP_FEATURES = [
    'age',
    'Pre_KPS',
    'DYSLIPID',
    'LUNG',
    'PRE_SEIZURE',
    'OP_TYPE'
]

# ตรวจว่ามีโมเดลที่ฝึกแล้ว
for name in MODEL_NAMES:
    if name not in fitted_models:
        raise ValueError(f'ไม่พบ fitted model: {name}')

    if list(fitted_models[name].feature_names_in_) != APP_FEATURES:
        raise ValueError(
            f'{name}: predictors ไม่ตรงกับโมเดลล่าสุด 6 ตัว'
        )

# ดึงชนิดผ่าตัดที่แต่ละโมเดลเคยเห็นใน training
operation_levels_by_model = {}

for name in MODEL_NAMES:
    encoder = (
        fitted_models[name]
        .named_steps['preprocess']
        .named_transformers_['operation']
        .named_steps['encoder']
    )

    operation_levels_by_model[name] = set(
        encoder.categories_[0].tolist()
    )

# แสดงเฉพาะชนิดผ่าตัดที่ทุกโมเดลรองรับ
common_operations = set.intersection(
    *operation_levels_by_model.values()
)

if not common_operations:
    raise ValueError('ไม่พบชนิดผ่าตัดที่ทุกโมเดลรองรับ')

operation_values = sorted(
    common_operations,
    key=lambda value: str(value)
)

# หาก training ใช้รหัสตัวเลข สามารถเพิ่มชื่อเพื่อแสดงใน dropdown
# key ต้องตรงกับค่าที่อยู่ใน encoder เช่น '1' ไม่ใช่ 1
# ชื่อชนิดผ่าตัดตามรหัสต้นฉบับ
OP_LABELS = {
    1: 'Craniotomy with tumor removal',
    2: 'Burr hole with biopsy',
    3: 'Craniectomy',
    4: 'Suboccipital approach',
    5: 'Retrosigmoid approach',
    6: 'Combined petrosectomy / Skull base surgery',
    7: 'Transsphenoidal approach',
    8: 'Combined transsphenoidal and transcranial approaches',
    9: 'Awake craniotomy',
    10: 'Ventriculostomy',
    11: 'Third ventriculostomy with biopsy',
    12: 'VP shunt'
}

# common_operations มาจาก fitted encoders ในโค้ดก่อนหน้า
# รองรับรหัส เช่น '1', '1.0', 1 หรือ 1.0
def operation_code(value):
    try:
        number = float(value)
    except (TypeError, ValueError):
        raise ValueError(
            f'หมวดหมู่ในโมเดลเป็นข้อความ "{value}" '
            'กรุณาตรวจสอบก่อนจับคู่กับรหัส 1–12'
        )

    if not number.is_integer() or int(number) not in OP_LABELS:
        raise ValueError(f'ไม่พบ label สำหรับรหัส: {value}')

    return int(number)


operation_values = sorted(
    common_operations,
    key=operation_code
)

# (ข้อความที่แสดง, ค่าจริงที่โมเดลใช้)
operation_choices = [
    (
        f'{operation_code(value)}. '
        f'{OP_LABELS[operation_code(value)]}',
        value
    )
    for value in operation_values
]

def predict_cost(
    algorithm,
    age,
    kps,
    dyslipid,
    lung,
    pre_seizure,
    operation
):
    if algorithm not in fitted_models:
        raise gr.Error('กรุณาเลือก algorithm')

    # ตรวจข้อมูลกรอก
    if age is None or kps is None:
        raise gr.Error('กรุณากรอกอายุและ KPS')

    if not (
        np.isfinite(float(age))
        and np.isfinite(float(kps))
    ):
        raise gr.Error('อายุและ KPS ต้องเป็นตัวเลขที่ใช้ได้')

    if not 0 <= float(age) <= 120:
        raise gr.Error('อายุต้องอยู่ระหว่าง 0–120 ปี')

    if not 0 <= float(kps) <= 100:
        raise gr.Error('KPS ต้องอยู่ระหว่าง 0–100')

    if any(
        value not in [0, 1]
        for value in [dyslipid, lung, pre_seizure]
    ):
        raise gr.Error('กรุณาระบุข้อมูลโรคร่วมและอาการให้ครบ')

    if operation not in operation_levels_by_model[algorithm]:
        raise gr.Error(
            'ชนิดผ่าตัดนี้ไม่พบใน training ของโมเดล '
            'จึงไม่สามารถทำนายได้'
        )

    # ส่งข้อมูลดิบเข้า fitted pipeline
    # pipeline จะทำ preprocessing ด้วยค่าจาก training เดิม
    input_data = pd.DataFrame({
        'age': [float(age)],
        'Pre_KPS': [float(kps)],
        'DYSLIPID': [float(dyslipid)],
        'LUNG': [float(lung)],
        'PRE_SEIZURE': [float(pre_seizure)],
        'OP_TYPE': [operation]
    })[APP_FEATURES]

    predicted_cost = float(
        fitted_models[algorithm].predict(input_data)[0]
    )

    if not np.isfinite(predicted_cost):
        raise gr.Error('โมเดลไม่สามารถให้ค่าทำนายที่ใช้ได้')

    if predicted_cost < 0:
        return (
            f'### {algorithm}\n'
            'โมเดลให้ค่าทำนายติดลบ จึงไม่สามารถใช้เป็น'
            'ประมาณการค่าใช้จ่ายสำหรับข้อมูลนี้ได้'
        )

    return (
        f'### ค่าใช้จ่ายรวมที่ทำนาย\n'
        f'## {predicted_cost:,.0f} บาท\n'
        f'Algorithm: **{algorithm}**\n\n'
        'เป็นค่าประมาณเพื่อการวิจัย ไม่ใช่ใบเสนอราคา'
        'หรือช่วงทำนายรายบุคคล'
    )


with gr.Blocks(title='Brain Tumor Surgery Cost Prediction') as demo:

    gr.Markdown(
        '# Brain Tumor Surgery Cost Prediction\n'
        'เลือก algorithm และกรอกข้อมูลก่อนผ่าตัด '
        'เพื่อประมาณค่าใช้จ่ายรวม'
    )

    algorithm_input = gr.Dropdown(
        choices=MODEL_NAMES,
        value='OLS',
        label='Prediction algorithm',
        type='value'
    )

    with gr.Row():
        age_input = gr.Number(
            label='Age (years)',
            value=50,
            minimum=0,
            maximum=120
        )

        kps_input = gr.Slider(
            minimum=0,
            maximum=100,
            value=80,
            step=10,
            label='Preoperative KPS'
        )

    with gr.Row():
        dyslipid_input = gr.Radio(
            choices=[('ไม่มี / No', 0), ('มี / Yes', 1)],
            value=0,
            label='Dyslipidemia'
        )

        lung_input = gr.Radio(
            choices=[('ไม่มี / No', 0), ('มี / Yes', 1)],
            value=0,
            label='LUNG — eg.COPD,Asthma'
        )

        seizure_input = gr.Radio(
            choices=[('ไม่มี / No', 0), ('มี / Yes', 1)],
            value=0,
            label='Preoperative seizure'
        )

    operation_input = gr.Dropdown(
    choices=operation_choices,
    value=None,
    label='Planned operation type',
    type='value',
    allow_custom_value=False
)

    predict_button = gr.Button(
        'Predict cost',
        variant='primary'
    )

    result_output = gr.Markdown()

    predict_button.click(
        fn=predict_cost,
        inputs=[
            algorithm_input,
            age_input,
            kps_input,
            dyslipid_input,
            lung_input,
            seizure_input,
            operation_input
        ],
        outputs=result_output
    )

    gr.Markdown(
        'ใช้เพื่อการวิจัย โดยชนิดผ่าตัดต้องเป็นแผนที่ทราบก่อนผ่าตัด '
        'และใช้นิยามค่าใช้จ่ายเดียวกับข้อมูลที่ใช้ฝึกโมเดล '
        'ไม่ต้องกรอกชื่อหรือ HN'
    )

#Step 3 - Gradio in Colab

In [ ]:
demo.launch(
    share=True,
    debug=True
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://206e282d4f7e612607.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
from app import build_app
demo = build_app("results/final_model.json")
demo.launch(share=True, debug=True)